In [4]:
# using Pkg
# Pkg.add("JuMP")
# Pkg.add("Gurobi")
# Pkg.add("LightGraphs")
# Pkg.add("DataFrames")
# Pkg.add("Query")
# Pkg.add("CSV")
# Pkg.add("TimerOutputs")
# Pkg.add("Dates")
# Pkg.add("Polynomials")

#Ready for upload

global A1 = 0 # 1=Select arc having the largest uncertainty , 0=Select arc using Lemma2
global A2 = 1 # 1=Partition once per cell , 0=Partition multiple per cell
global A3 = 0# 1=Split at mean base cost , 0=Split using SA if possible
global A4 = 1 # 1=Frequent solve MP
#If running A5 = 0, do not use this file, use LazyAlg.jl instead
global A5 = 1 # 1=Regular opt model

include("_test_functionLoadSharedFiles.jl")

Set parameter Username
Academic license - for non-commercial use only - expires 2024-05-29
Ins N50_16 Running...04:40:08


0

In [5]:
#Setting constraint for start node
# outgoing = findall(edge[:,1].== origin)

# If we want to add # in Gurobi, then we have to turn of Gurobi's own Cuts 
# h1 = Model(() -> Gurobi.Optimizer(gurobi_env))
# h1.setParam("OutputFlag", 0)
# set_optimizer_attribute(h1, "OutputFlag", 0)

# @variable(h1, 1 >= y_h[1:Len]>=0)
#@variable(h, q[1:Len]>=0)

#Setting constraints for remaining none-sink/start nodes
# @constraint(h1, sum(y_h[k] for k in outgoing) == 1)
# for i in all_nodes
#     global outgoing
#     if i != destination && i != origin
#         incoming = findall(edge[:,2].== i)
#         outgoing = findall(edge[:,1].== i)
#         @constraint(h1, sum(-y_h[k] for k in outgoing) + sum(y_h[k] for k in incoming) == 0)
#     end
# end


#MAIN PROGRAM:
# if A2==1 #Partition once per cell
df_constraints = DataFrame(NUM = Int[], CELL = Int[], Y = Array[], SP = Float64[])
df_cell = DataFrame(CELL = Int[], Y = Array[], Y_Lk = Array[], g = Float64[], h = Float64[], gL = Float64[], LB = Array[], UB = Array[], PROB = Float64[], PI = Array[])

# M = cU_orig - cL_orig
c = (cU_orig + cL_orig)/2  
yy, SP_init, SP_init, T, pred, label, path = gx_bound(c, c, edge) #y, gx, SP, T, pred, label, path
# print("label ", label)
push!(df_cell, (1, yy,yy, SP_init, 0, 0, cL_orig, cU_orig, 1,label))
push!(df_constraints, (1, 1,yy,SP_init))
# else #Partition multiple per cell
#     df_constraints = DataFrame(NUM = Int[], CELL = Int[], Y = Array[], SP = Float64[], STAT= Int[])
# df_cell = DataFrame(CELL = Int[], Y = Array[], Y_Lk = Array[], g = Float64[], h = Float64[], gL = Float64[], LB = Array[], UB = Array[], PROB = Float64[])
#     push!(df_cell, (1, yy,yy, SP_init, 0, 0, cL_orig, cU_orig, 1))
#     push!(df_constraints, (1, 1,yy,SP_init,1))
# end



##println(f,"MASTER PROBLEM==========================================================================================")

MP_obj = 0.0
zNum = 200000
cRefNum = 2000000
if A1==1
    zNum = 2000000
    cRefNum = 20000000
end
m = Model(() -> Gurobi.Optimizer(gurobi_env)) # If we want to add # in Gurobi, then we have to turn of 
# set_optimizer_attribute(m, "OutputFlag", 0)    #Gurobi's own Cuts 
# println("1")
@variable(m, x[1:Len], Bin)
# @variable(m, α)
@variable(m, 1e6 >= z[1:zNum] >= 0)
# @constraintref constr[1:200000]
# @ConstrRef constr[1:200000]

# constr = Array{JuMP.JuMPArray{JuMP.ConstraintRef,1,Tuple{Array{Int64,1}}}}()

constr = Array{JuMP.ConstraintRef}(undef, cRefNum)
@constraint(m, sum(x[i] for i=1:Len) == b) 
constr[1] = @constraint(m, z[1] <= SP_init + sum(yy[i]*x[i]*d[i] for i=1:Len) )
@objective(m, Max, sum(p[i]*z[i] for i = 1:length(p)) )#w - sum(s[k] for k=1:length(s))/length(s) )
include("functionSetGlobalVar_MP.jl")
if A2 == 1
    if A4 == 1 #1=Frequent solve MP
        include("A2_1.jl") #Partition once per cell
    else
        println("A4_0")
        include("A4_0.jl")
    end
else
    include("A2_0.jl") #Partition multiple per cell
end
# println("con_num " , con_num)
# println("constr ", constr[1:con_num])
# println("z_now ", z_now[1:newCell])
total_time = time() - start
set = string(A1)*string(A2)*string(A3)*string(A4)*string(A5)
println("Alg_"*set*"_"*dataSet, "; Ins ", Ins, "; Time ", total_time, "; MP_obj ", MP_obj, "; x_now ", findall(x_now.==1),"; Cells ", nrow(df_cell), "; Iter ", iter)#, "; W ", LB_w, "; Cuts ", numConv)
h_val = df_cell.h
p_val = df_cell.PROB

println("LB = ", sum(h_val[k]*p_val[k] for k=1:newCell))

# timesFile = open("./OutputFile/Alg_"*set*"_"*dataSet*".txt", "a")
# println(timesFile, dataSet, "; Ins ", Ins, "; Time ", total_time, "; MP_obj ", MP_obj, "; x_now ", findall(x_now.==1),"; Cells ", nrow(df_cell), "; Iter ", iter)#, "; W ", LB_w, "; Cuts ", numConv)
# close(timesFile)
# println(LB_w + β)
# println("\007")

┌ Warning: Assignment to `c` in soft scope is ambiguous because a global variable by the same name exists: `c` will be treated as a new local. Disambiguate by using `local c` to suppress this warning or `global c` to assign to the existing global variable.
└ @ ~/Library/CloudStorage/GoogleDrive-di.hoai.nguyen@gmail.com/Other computers/My Laptop/Documents/GitHub/Paper5/A2_1.jl:33
┌ Warning: Assignment to `T` in soft scope is ambiguous because a global variable by the same name exists: `T` will be treated as a new local. Disambiguate by using `local T` to suppress this warning or `global T` to assign to the existing global variable.
└ @ ~/Library/CloudStorage/GoogleDrive-di.hoai.nguyen@gmail.com/Other computers/My Laptop/Documents/GitHub/Paper5/A2_1.jl:34
┌ Warning: Assignment to `pred` in soft scope is ambiguous because a global variable by the same name exists: `pred` will be treated as a new local. Disambiguate by using `local pred` to suppress this warning or `global pred` to assign 


Iter : 1 ; MP_obj = 503.0 ; time 0.4371061325073242; 1/1
x = [1, 2, 3, 4, 5, 127, 211]

Iter : 2 ; MP_obj = 503.0 ; time 0.6112592220306396; 1/1
x = [1, 2, 3, 12, 115, 127, 211]

Iter : 3 ; MP_obj = 503.0 ; time 0.7861151695251465; 1/1
x = [1, 3, 115, 127, 186, 211, 247]

Iter : 4 ; MP_obj = 502.0 ; time 0.9593310356140137; 1/1
x = [1, 3, 53, 115, 127, 186, 211]

Iter : 5 ; MP_obj = 500.0 ; time 1.1327471733093262; 1/1
x = [1, 3, 36, 39, 115, 127, 211]
Cell 1. gx = 500.0; hx = 482.5

Iter : 6 ; MP_obj = 500.0 ; time 1.756829023361206; 2/2
x = [1, 3, 36, 39, 115, 127, 211]
Cell 1. gx = 500.0; hx = 482.5
Cell 2. gx = 500.0; hx = 487.5

Iter : 7 ; MP_obj = 498.675 ; time 1.9600961208343506; 4/4
x = [1, 3, 36, 39, 115, 127, 211]
Cell 1. gx = 498.0; hx = 482.5
Cell 2. gx = 495.5; hx = 487.5
Cell 3. gx = 500.0; hx = 484.5
Cell 4. gx = 500.0; hx = 492.0

Iter : 8 ; MP_obj = 497.8638888888889 ; time 2.1857261657714844; 8/8
x = [1, 3, 11, 115, 127, 186, 211]

Iter : 9 ; MP_obj = 497.5361111111

LoadError: LoadError: Result index of attribute MathOptInterface.ObjectiveValue(1) out of bounds. There are currently 0 solution(s) in the model.
in expression starting at /Users/dinguyen/Library/CloudStorage/GoogleDrive-di.hoai.nguyen@gmail.com/Other computers/My Laptop/Documents/GitHub/Paper5/A2_1.jl:1